# Is your house worth it's weight in gold?

Now we've calculated how much a given house rose in value over time - the question is, would you have been better off buying gold?

In the `data` folder, we have the daily price of one ounce of gold in USD, called `XAUUSD` in trading terms (e`X`change rate of `AU` aka Gold to `USD`)

We want to find out what would have happened if instead of buying a house, we would have bought an equivalent sum of gold. Finally we want to compare what the average profit would be if everyone had invested in gold instead of buying a house.

For simplicity, use the daily Closing price.

```{note}
Since gold is priced in USD, and our houseprices are in GBP, there is also a USD/GBP daily prices in the `data/fx` folder to convert between currencies.
```

Breaking it down:

1. Load daily gold prices into Iceberg - for bonus points, apply a yearly partitioning scheme
2. Load exchange rates into Iceberg - for bonus points, apply a yearly partitioning scheme
3. Convert daily gold prices from USD to GBP
4. For each row in our `profits` table, calculate the equivalent amount of gold they would have been able to purchase on that date
5. Calculate the total value of that amount of gold on the sell date
6. Calculate the profit of our gold trade
7. Compare to Gold profit with Housing profit

In [25]:
import polars as pl
from utils import catalog, engine
from pyiceberg.schema import Schema, NestedField
from pyiceberg.types import DecimalType, DateType, StringType
from pyiceberg.partitioning import PartitionSpec, YearTransform, PartitionField
from IPython.display import display
pl.Config.set_thousands_separator(",")

polars.config.Config

## 1. Load daily gold prices into Iceberg
Start by creating the table in Iceberg. To organize things a bit better, I'm creating a new namespace `commodities` - we could imagine putting a table for oil prices or silver in here

In [26]:
catalog.create_namespace_if_not_exists("commodities")

gold_schema = Schema(
    NestedField(1, "date", DateType(), required=True, doc="Day of recorded price"),
    NestedField(
        2,
        "price",
        DecimalType(precision=38, scale=2),
        required=True,
        doc="Price in USD of one ounce of gold",
    ),
    identifier_field_ids=[1],
)

In [27]:
gold_prices_t = catalog.create_table_if_not_exists(
    "commodities.gold",
    schema=gold_schema,
    partition_spec=PartitionSpec(
        PartitionField(
            source_id=1, field_id=1, transform=YearTransform(), name="date_year"
        )
    ),
)

Next, read in the CSV, picking out the two columns we care about, remembering to convert to the correct schema, 

In [28]:
gold_prices = (
    pl.scan_csv("data/gold/daily_gold_prices.csv", separator=";", try_parse_dates=True)
    .select(pl.col("Date").alias("date"), pl.col("Close").alias("price"))
    .collect()
)
gold_prices

date,price
datetime[μs],f64
2004-06-11 00:00:00,384.1
2004-06-14 00:00:00,382.8
2004-06-15 00:00:00,388.6
2004-06-16 00:00:00,383.8
2004-06-17 00:00:00,387.6
…,…
2025-01-28 00:00:00,"2,763.17"
2025-01-29 00:00:00,"2,759.68"
2025-01-30 00:00:00,"2,794.06"


In [29]:
gold_prices_t.append(gold_prices.to_arrow().cast(gold_schema.as_arrow()))

# 2. Load daily exchange rates into Iceberg

Similar process - create a new namespace and load the fx rates. Since this is technically a table of currency pairs, we can make the schema a bit more future-proof

In [30]:
catalog.create_namespace_if_not_exists("fx")

fx_schema = Schema(
    NestedField(1, "date", DateType(), required=True, doc="Day of recorded price"),
    NestedField(
        2,
        "from",
        StringType(),
        required=True,
        doc="From currency code of the currency pair",
    ),
    NestedField(
        3,
        "to",
        StringType(),
        required=True,
        doc="To currency code of the currency pair",
    ),
    NestedField(
        4,
        "exchange_rate",
        DecimalType(precision=38, scale=4),
        required=True,
        doc="Exchange rate of the currency pair",
    ),
    identifier_field_ids=[1, 2, 3],
)

Next, reading in the CSV file and transforming slightly to conform to our schema

In [31]:
fx_table = catalog.create_table_if_not_exists("fx.rates", schema=fx_schema)

In [32]:
fx_rates = (
    pl.scan_csv("data/fx/USD_GBP.csv", try_parse_dates=True)
    .with_columns(pl.lit("USD").alias("from"), pl.lit("GBP").alias("to"))
    .select(
        pl.col("date"),
        pl.col("from"),
        pl.col("to"),
        pl.col("close").cast(pl.Decimal(38, 4)).alias("exchange_rate"),
    )
    .collect()
)
fx_rates

date,from,to,exchange_rate
date,str,str,"decimal[38,4]"
2025-05-16,"""USD""","""GBP""",0.7529
2025-05-15,"""USD""","""GBP""",0.7516
2025-05-14,"""USD""","""GBP""",0.7540
2025-05-13,"""USD""","""GBP""",0.7516
2025-05-12,"""USD""","""GBP""",0.7590
…,…,…,…
2006-03-07,"""USD""","""GBP""",0.5760
2006-03-06,"""USD""","""GBP""",0.5713
2006-03-03,"""USD""","""GBP""",0.5696


In [33]:
fx_table.append(fx_rates.to_arrow().cast(fx_schema.as_arrow()))

## Business Logic
Next comes the business logic - the actual work of figuring out how much gold we could have bought and how much that would have sold for

In [36]:
sql = """
-- Convert USD gold prices into GBP denominated prices
with gold_prices as (
    select commodities.gold.date as gold_date, 
           commodities.gold.price * fx.rates.exchange_rate as gold_price
    from commodities.gold
    join fx.rates on fx.rates.date = commodities.gold.date
    where commodities.gold.date >= DATE '2023-01-01'  -- Add date filter if appropriate
), 
filtered_profits as (
    select * from housing.profits 
    where first_day >= DATE '2023-01-01'  -- Match the date filter
),
gold_purchase as (
    select address_id, 
           first_price / gold_price as purchased_gold,
           first_day,
           last_day
    from filtered_profits
    join gold_prices on gold_date = filtered_profits.first_day
), 
gold_sell as (
    select gp.address_id,
           cast((gp.purchased_gold * gold_prices.gold_price) - fp.first_price as DECIMAL(38, 2)) as gold_profit,
           cast(fp.profit as DECIMAL(38, 2)) as house_profit
    from gold_purchase gp
    join filtered_profits fp on fp.address_id = gp.address_id
    join gold_prices on gold_prices.gold_date = gp.last_day
)
select * from gold_sell
"""

In [37]:
gold_vs_house_profits = pl.read_database(sql, engine)
gold_vs_house_profits

address_id,gold_profit,house_profit
str,"decimal[38,2]","decimal[38,2]"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.39",0.00
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.39",0.00
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.39",0.00
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.39",0.00
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.39",0.00
…,…,…
"""523EA4049BE7C657617E05D55E84B6…","10,636.74","747,275.00"
"""523EA4049BE7C657617E05D55E84B6…","10,636.74","747,275.00"
"""523EA4049BE7C657617E05D55E84B6…","10,636.74","747,275.00"


Now we have a per-address calculation, let's summarize the results

In [38]:
gold_vs_house_profits.select(pl.all().exclude("address_id")).describe()

statistic,gold_profit,house_profit
str,f64,f64
"""count""",5.721192e6,5.721192e6
"""null_count""",0.0,0.0
"""mean""",781.73078,"68,029.004587"
"""std""","13,406.771549","858,620.957125"
"""min""","-173,440.68","-722,888.0"
"""25%""","-2,902.53",0.0
"""50%""",245.19,"25,000.0"
"""75%""","4,873.13","65,000.0"
"""max""","133,224.19",2.4992e7


How many percent would have done better buying gold than a house?

In [39]:
with pl.Config(set_tbl_rows=100):
    summary_df = (
        gold_vs_house_profits.select(
            pl.col("gold_profit")
            .sub(pl.col("house_profit"))
            .qcut(100, labels=[f"Q{i + 1}" for i in range(100)], include_breaks=True)
        )
        .unnest("gold_profit")
        .unique()
        .sort("breakpoint")
    )
    display(summary_df)

breakpoint,category
f64,cat
"-272,954.01","""Q1"""
"-220,522.12","""Q2"""
"-180,180.57","""Q3"""
"-158,757.23","""Q4"""
"-143,741.29","""Q5"""
"-135,952.42","""Q6"""
"-124,655.77","""Q7"""
"-119,437.55","""Q8"""
"-111,505.95","""Q9"""


# Exercise: What does it look like in your region?

This is the picture for all of the UK - what does it look like for your region?

In [41]:
with pl.Config(set_tbl_rows=100):
    summary_df = (
        gold_vs_house_profits
        # .filter(pl.col("county") == "KENT")  # Add this line to filter by county
        .select(
            pl.col("gold_profit")
            .sub(pl.col("house_profit"))
            .qcut(100, labels=[f"Q{i + 1}" for i in range(100)], include_breaks=True)
        )
        .unnest("gold_profit")
        .unique()
        .sort("breakpoint")
    )
    display(summary_df)

breakpoint,category
f64,cat
"-272,954.01","""Q1"""
"-220,522.12","""Q2"""
"-180,180.57","""Q3"""
"-158,757.23","""Q4"""
"-143,741.29","""Q5"""
"-135,952.42","""Q6"""
"-124,655.77","""Q7"""
"-119,437.55","""Q8"""
"-111,505.95","""Q9"""
